In [44]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [45]:
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

In [46]:
train_df.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [47]:
train_df.shape

(2000, 8)

In [48]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


In [49]:
train_df.isnull().sum()

id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

In [50]:
train_df['answer'].value_counts()

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

In [51]:
from datasets import load_dataset

dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)

In [52]:
train_dataset = dataset["train"]

def combine_text(example):
    example["combined_text"] = example["prompt"] + " " + example["A"]
    return example

train_dataset = train_dataset.map(combine_text)

print(train_dataset[51]["combined_text"])
print(len(train_dataset[51]["combined_text"]))

Determine the correct option: What is the reason behind the designation of Class L dwarfs, and what is their color and composition? among the listed options. Class L dwarfs are hotter than M stars and are designated L because L is the remaining letter alphabetically closest to M. They are bright blue in color and are brightest in ultraviolet. Their atmosphere is hot enough to allow metal hydrides and alkali metals to be prominent in their spectra. Some of these objects have masses large enough to support hydrogen fusion and are therefore stars, but most are of substellar mass and are therefore brown dwarfs.
614


In [53]:
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased", output_attentions=True)
print(tokenizer.vocab_size)
print(tokenizer.get_vocab()["[SEP]"])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


30522
102


In [54]:
prompts = list(train_dataset["prompt"])

encoded = tokenizer(
    prompts,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

print(encoded["input_ids"].shape)

torch.Size([2000, 128])


In [55]:
prompt = train_dataset[0]["prompt"]
print(prompt)

inputs = tokenizer(prompt, return_tensors="pt")
outputs = model(**inputs)
print(outputs.last_hidden_state.shape)

Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.
torch.Size([1, 31, 768])


In [56]:
cls_embedding = outputs.last_hidden_state[0, 0]
answer = cls_embedding[:5].sum().item()

print(round(answer, 4))

-1.2001


In [57]:
text = "Light-ion fusion is a technique."

inputs_ = tokenizer(text, return_tensors="pt")

tokens = tokenizer.convert_ids_to_tokens(inputs_["input_ids"][0])

for i, token in enumerate(tokens):
    print(i, token)

outputs = model(**inputs_)

last_layer = outputs.attentions[-1]
head0 = last_layer[0, 0]


0 [CLS]
1 light
2 -
3 ion
4 fusion
5 is
6 a
7 technique
8 .
9 [SEP]


In [58]:
fusion_index = 4

weight = head0[0, fusion_index].item()

print(round(weight, 4))

0.1025


In [59]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

prompt = train_dataset[0]["prompt"]
option_b = train_dataset[0]["B"]

prompt_embedding = model.encode(prompt, convert_to_tensor=True)
option_b_embedding = model.encode(option_b, convert_to_tensor=True)

similarity = util.cos_sim(prompt_embedding, option_b_embedding)

score = similarity.item()

print(round(score, 4))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


0.7658


In [60]:
def apk(actual, predicted, k=3):
    if len(predicted) > k:
        predicted = predicted[:k]

    score = 0.0

    for i, p in enumerate(predicted):
        if p == actual:
            score = 1.0 / (i + 1)
            break

    return score


def mapk(actuals, predictions, k=3):
    return sum(apk(a, p, k) for a, p in zip(actuals, predictions)) / len(actuals)

In [61]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf_predictions = []

for row in train_dataset:

    texts = [
        row["prompt"],
        row["A"],
        row["B"],
        row["C"],
        row["D"],
        row["E"]
    ]

    vectorizer = TfidfVectorizer(stop_words="english")

    X = vectorizer.fit_transform(texts)

    similarities = cosine_similarity(X[0], X[1:]).flatten()

    labels = ["A", "B", "C", "D", "E"]

    ranking = sorted(
        zip(labels, similarities),
        key=lambda x: x[1],
        reverse=True
    )

    tfidf_predictions.append([x[0] for x in ranking[:3]])

In [62]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

minilm_predictions = []

for row in train_dataset:

    prompt = row["prompt"]

    options = [
        row["A"],
        row["B"],
        row["C"],
        row["D"],
        row["E"]
    ]

    prompt_embedding = model.encode(
        prompt,
        convert_to_tensor=True
    )

    option_embeddings = model.encode(
        options,
        convert_to_tensor=True
    )

    similarities = util.cos_sim(
        prompt_embedding,
        option_embeddings
    )[0]

    labels = ["A", "B", "C", "D", "E"]

    ranking = sorted(
        zip(labels, similarities.tolist()),
        key=lambda x: x[1],
        reverse=True
    )

    minilm_predictions.append(
        [x[0] for x in ranking[:3]]
    )

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [63]:
answers = train_dataset["answer"]

tfidf_map3 = mapk(
    answers,
    tfidf_predictions
)

minilm_map3 = mapk(
    answers,
    minilm_predictions
)

print(tfidf_map3)
print(minilm_map3)

0.36691666666666667
0.4230833333333333


In [64]:
count = 0

for actual, tfidf_pred, minilm_pred in zip(
    answers,
    tfidf_predictions,
    minilm_predictions
):

    if (
        actual not in tfidf_pred
        and
        actual in minilm_pred
    ):
        count += 1

print(count)

374


In [66]:
from transformers import pipeline

classifier = pipeline("zero-shot-classification")

prompt = train_dataset[1]["prompt"]

candidate_labels = [
    train_dataset[1]["A"],
    train_dataset[1]["B"],
    train_dataset[1]["C"]
]

result = classifier(
    prompt,
    candidate_labels
)

top_score = result["scores"][0]

print(round(top_score, 4))

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

0.4575


In [72]:
result_multi = classifier(
    prompt,
    candidate_labels,
    multi_label=True
)

sum_multi = sum(result_multi["scores"])
print(sum_multi)

0.0005096072964079212


In [69]:
softmax_sum = sum(result["scores"])
sigmoid_sum = sum(result_multi["scores"])

difference = abs(softmax_sum - sigmoid_sum)

print(round(difference, 4))

0.9995


In [79]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

row = train_dataset[0]

prompt = (
    f"Question: {row['prompt']}. "
    f"Is the correct answer A: {row['A']} "
    f"or B: {row['B']}? "
    f"Answer with just the letter A or B."
)

inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_new_tokens=5
)

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(answer)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


B
